# Live session run — **streaming** extraction (RAM-independent)

Full CNMFe pipeline — fused AVI → motion correction → **true T-streaming
extraction** — for a session that may be too large to hold in RAM, with every
knob that affects streaming speed explained.

**Stream, or load into RAM?**

| | how | when |
|---|---|---|
| **In-memory** | `movie = np.asarray(mc); model.fit_extract(movie)` | the corrected movie fits in RAM (`T·H·W·4` bytes). One pass at memory speed — **always faster**. |
| **Streaming** *(this notebook)* | `model.fit_extract(mc, output_dir=…)` | the movie does **not** fit in RAM, or you want a reproducible disk-handoff. The movie stays on disk as a pixel-major `Y_flat` store; extraction reads it in ~5–6 passes. Peak RAM is independent of `T`. |

Streaming is **IO-bound**: the cost is how fast the on-disk `Y_flat` store can be
read. That is exactly what the knobs in section 6 control. `fit_extract` prints a
**per-stage timing summary** so you can see where the time actually goes.

## 1. Imports

In [ ]:
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Make the project root importable so 'concat_avis_to_zarr' resolves anywhere.
PROJECT_ROOT = Path('/home/fs539/code/simpler_cnmfe')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from minicnmfe.io import (
    open_zarr,
    open_zarr_pixel_major,
    stage_zarr_to_local,
    transpose_zarr_to_pixel_major,
)
from minicnmfe.pipeline import CNMFe, CNMFeParams
from concat_avis_to_zarr import _count_and_shape, _numeric_key

## 2. Session paths + downsample + streaming scratch dir

`YFLAT_DIR` is the streaming-specific addition: the directory where the
pixel-major `Y_flat` store is written. **On a network mount, point it at a local
SSD / tmpfs** — the transpose reads `mc.zarr` from the network *once*, then every
BCD pass reads `Y_flat` locally. Set it to `None` to keep `Y_flat` next to the
results (all-network: simplest, slowest).

In [ ]:
# --- EDIT ME -----------------------------------------------------------------
SESSION_RAW = Path(
    '/media/server/archive/projects/2023_intercontext/ToneComparison/'
    'data/0_raw/20260520_m0000159_som_1145/miniscope_video'
)
SSUB = 2    # spatial bin factor  (1 = no spatial downsampling)
TSUB = 2    # temporal bin factor (1 = no temporal downsampling)

# Streaming scratch: a LOCAL SSD/tmpfs dir for the pixel-major Y_flat store.
# None => write Y_flat next to the results (all-network; simplest but slowest).
YFLAT_DIR = Path('/tmp/cnmfe_yflat') / SESSION_RAW.parent.name
# YFLAT_DIR = None
# -----------------------------------------------------------------------------

def _swap_stage(p: Path, new_stage: str) -> Path:
    parts = list(p.parts)
    try:
        i = parts.index('0_raw')
    except ValueError as exc:
        raise ValueError(f"Expected '0_raw' in {p}") from exc
    parts[i] = new_stage
    return Path(*parts)

SESSION_PREPROC = _swap_stage(SESSION_RAW, '1_preprocessed')
SESSION_PROC    = _swap_stage(SESSION_RAW, '2_processed')

DOWNSAMPLE  = (SSUB > 1 or TSUB > 1)
RAW_AVI_DIR = SESSION_RAW
MC_ZARR     = SESSION_PREPROC / 'mc.zarr'      # the only zarr written (fused)
MC_PLOTS    = SESSION_PROC / 'mc_plots'
RESULTS_DIR = SESSION_PROC / 'results'

SESSION_PREPROC.mkdir(parents=True, exist_ok=True)
SESSION_PROC.mkdir(parents=True, exist_ok=True)
MC_PLOTS.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Raw AVIs   :', RAW_AVI_DIR)
print(f'Downsample : ssub={SSUB} tsub={TSUB}  ({"on" if DOWNSAMPLE else "off"})')
print('MC zarr   ->', MC_ZARR)
print('Y_flat dir ->', YFLAT_DIR, '(local scratch)' if YFLAT_DIR else '(= results dir)')
print('Results   ->', RESULTS_DIR)
assert RAW_AVI_DIR.is_dir(), f'Raw folder does not exist: {RAW_AVI_DIR}'

## 3. Discover AVIs

In [ ]:
avis = sorted(RAW_AVI_DIR.glob('*.avi'), key=_numeric_key)
avis = [p for p in avis if _numeric_key(p) >= 0]
assert avis, f'No numerically-named AVIs in {RAW_AVI_DIR}'

print(f'{len(avis)} AVI file(s):')
total = 0
H_raw = W_raw = None
for p in avis:
    n, h, w = _count_and_shape(p)
    total += n
    if H_raw is None:
        H_raw, W_raw = h, w
print(f'  TOTAL {total} frames  {H_raw}x{W_raw}')

## 4. Parameters (native units)

In [ ]:
# NATIVE (full-resolution) units; rescaled to the downsampled grid via
# params.downscaled(SSUB, TSUB) in the next cell. params_eff drives MC + extraction.
params = CNMFeParams(
    # --- motion correction ---
    max_shift=(20, 20),
    upsample_factor=10,
    mc_n_iter=1,                   # fused AVI->MC path supports 1 only
    mc_gSig_filt=7,                # 1p high-pass sigma; None for 2p
    mc_batch_size=1000,
    mc_template_max_frames=2000,
    mc_output_dtype='float32',
    # --- CNMFe extraction ---
    sigma=5.0,
    min_corr=0.8,
    min_pnr=10.0,
    n_iter_main=1,
    n_iter_temporal=2,
    merge_thr_corr=0.85,
    merge_thr_overlap=0.5,
    spatial_max_thr=0.1,
    spatial_max_iter=4000,
    n_jobs=-1,
    decay_time_ms=180,             # jGCaMP8m; see CLAUDE.md indicator table
    frame_rate_hz=20,
    g_prior_weight=0.95,
    global_ar=False,
)
print(f'native: sigma={params.sigma} max_shift={params.max_shift} '
      f'mc_gSig_filt={params.mc_gSig_filt}')

## 5. Motion correction — fused AVI → `mc.zarr` (+ optional downsample)

In [ ]:
# Rescale native params -> the (possibly downsampled) grid.
params_eff = params.downscaled(SSUB, TSUB)
if DOWNSAMPLE:
    print(f'downscaled: sigma {params.sigma}->{params_eff.sigma}, '
          f'min_pixel {params.min_pixel}->{params_eff.min_pixel}, '
          f'max_shift {params.max_shift}->{params_eff.max_shift}')

model_mc = CNMFe(params_eff)
t0 = time.time()
mc = model_mc.fit_mc_from_avis(
    RAW_AVI_DIR, output_dir=SESSION_PREPROC,
    ssub=SSUB, tsub=TSUB, skip_if_exists=True,
)
print(f'\nfused decode+MC elapsed: {time.time() - t0:.1f}s')

T_mc, H, W = mc.shape
print(f'mc shape={mc.shape}  chunks={mc.chunks}  dtype={mc.dtype}')

## 6. Streaming knobs — what to turn and why

Streaming makes **~5–6 full passes** over the on-disk `Y_flat` store
(`compute_W` ×2, `update_spatial`, `update_temporal` / `project_onto` ×several,
final `YrA`). Speed is dominated by **store IO**. Every knob below changes only
IO/compute speed — **never the extracted results** (pinned by the
streaming-vs-RAM equivalence tests).

### Store layout — `CNMFeParams.yflat_*`
| knob | default | turn it… |
|------|---------|----------|
| `yflat_dir` | `None` (next to results) | **network → local SSD/tmpfs.** Transpose reads `mc.zarr` from the network *once*; all BCD passes then read `Y_flat` locally. **Biggest network win.** |
| `yflat_pixel_chunk` | `512` | smaller (256) = less over-read for the 256-px read batches; larger (2048) = fewer chunk fetches for the 4096-px `project_onto`. |
| `yflat_time_chunk` | `None` = full `T` | keep `None` for the full-time read pattern; cap only if `pixel_chunk·T·4` bytes/chunk is too big for very long recordings. |
| `yflat_compression` | `True` | **network: keep `True`** (fewer bytes over the wire). **local SSD: try `False`** (no per-read decompression; costs ~`H·W·T·4` bytes of disk). |

### Passes & compute
| knob | default | effect |
|------|---------|--------|
| `n_jobs` | `1` → **set `-1`** | parallel per-batch reads + solves; hides network round-trip latency. |
| `bg_tsub` | `5` | temporal subsample for the ring `W` solve (cuts compute, not the read). |
| `n_iter_main` / `n_iter_temporal` | `2` / `2` | each removed iteration = one fewer full scan of the store. |
| `skip_first_deconv` | `True` | NNLS instead of OASIS on the first temporal pass. |
| `init_stride` / `sample_frames` | auto / `1000` | the *strided* init/noise sample (read from the 3-D `mc.zarr`, not `Y_flat`); controls init RAM + time. |

### Measuring
`fit_extract` prints a **per-stage wall-clock summary** at the end (and the
transpose prints `Done in Xs`). Change **one** knob, re-run, compare — don't guess.

### Should I even stream? (size / RAM check)

In [ ]:
ram_gb = T_mc * H * W * 4 / 1e9
print(f'corrected movie: {T_mc} x {H} x {W} float32  ~= {ram_gb:.1f} GB in RAM')
if ram_gb < 0.6 * 32:   # heuristic: comfortably under a typical 32 GB box
    print('  -> likely fits in RAM: the IN-MEMORY path is faster:')
    print('       movie = np.asarray(mc); model.fit_extract(movie)')
else:
    print('  -> probably too big for RAM: stream (below).')

pc = params_eff.yflat_pixel_chunk
tc = params_eff.yflat_time_chunk or T_mc
print(f'Y_flat chunk = {pc} px x {tc} frames = {pc * tc * 4 / 1e6:.1f} MB/chunk')

### Set the streaming knobs

In [ ]:
params_eff.yflat_dir         = None if YFLAT_DIR is None else str(YFLAT_DIR)
params_eff.yflat_pixel_chunk = 512       # pixels per Y_flat chunk
params_eff.yflat_time_chunk  = None      # None = full T (one chunk per pixel row)
params_eff.yflat_compression = True      # True on network; try False on local SSD
params_eff.n_jobs            = -1        # parallel reads hide network latency
params_eff.bg_tsub           = 5         # subsample time for the ring W solve
# params_eff.n_iter_main     = 1         # fewer BCD passes = fewer store scans

if YFLAT_DIR is not None:
    YFLAT_DIR.mkdir(parents=True, exist_ok=True)

print('streaming config:',
      f'yflat_dir={params_eff.yflat_dir}',
      f'pixel_chunk={params_eff.yflat_pixel_chunk}',
      f'time_chunk={params_eff.yflat_time_chunk}',
      f'compression={params_eff.yflat_compression}',
      f'n_jobs={params_eff.n_jobs}')

## 7. Streaming extraction (`fit_extract(mc, output_dir=…)` → auto-derive `Y_flat`)

Passing a **zarr** `movie` together with `output_dir` triggers the streaming
path: the pixel-major `Y_flat` store is auto-derived (under `params_eff.yflat_dir`
if set, else `output_dir`), then extraction reads it on disk — the 3-D `mc.zarr`
is touched only for the strided init sample, and the movie is **never** loaded
into RAM.

Watch the **per-stage timing summary** at the end. `transpose -> Y_flat` is your
one network read (when `yflat_dir` is local); the BCD stages below it should then
be local-disk speed.

In [ ]:
if (RESULTS_DIR / 'manifest.json').exists():
    print(f'results present in {RESULTS_DIR}; reloading (delete manifest.json to re-run)')
    model = CNMFe.load(RESULTS_DIR)
else:
    model = CNMFe(params_eff)
    t0 = time.time()
    # zarr movie + output_dir => STREAMING (no np.asarray; RAM independent of T).
    model.fit_extract(mc, output_dir=SESSION_PREPROC, evaluate=False)
    model.evaluate()                       # non-destructive component tagging
    elapsed = time.time() - t0
    print(f'\nextracted {model.A.shape[1]} components in {elapsed:.1f}s')
    print('^ read the per-stage timing summary above to see where the IO time went')

    model.save(RESULTS_DIR)
    (RESULTS_DIR / 'run_info.json').write_text(json.dumps({
        'session_raw'      : str(SESSION_RAW),
        'mc_zarr'          : str(MC_ZARR),
        'movie_shape'      : [int(T_mc), int(H), int(W)],
        'ssub'             : SSUB, 'tsub': TSUB,
        'streaming'        : True,
        'yflat_dir'        : params_eff.yflat_dir,
        'yflat_pixel_chunk': params_eff.yflat_pixel_chunk,
        'yflat_compression': params_eff.yflat_compression,
        'n_jobs'           : params_eff.n_jobs,
        'K_extracted'      : int(model.A.shape[1]),
        'wall_time_s'      : round(elapsed, 2),
    }, indent=2))
    print(f'saved -> {RESULTS_DIR}')

## 8. Alternative — build / stage `Y_flat` yourself (full control)

The auto-derive path above is the convenient default. For finer control — or
when you **already** have a `Y_flat` sitting on the network — drive the store
explicitly. (Both alternatives are commented out.)

In [ ]:
# --- A. Build the pixel-major store explicitly, then pass it as Y_flat_zarr ---
# Y_flat = transpose_zarr_to_pixel_major(
#     MC_ZARR, (YFLAT_DIR or SESSION_PREPROC) / 'Y_flat_pixel.zarr',
#     pixel_chunk=512, time_chunk=None, compression=True,   # tune freely
# )
# model = CNMFe(params_eff)
# model.fit_extract(mc, Y_flat_zarr=Y_flat, evaluate=False)
#
# --- B. You already have a Y_flat on the network -> stage it to local SSD once ---
# net_yflat   = SESSION_PREPROC / 'Y_flat_pixel.zarr'
# local_yflat = stage_zarr_to_local(net_yflat, '/tmp/cnmfe_yflat')   # one network copy
# model = CNMFe(params_eff)
# model.fit_extract(mc, Y_flat_zarr=local_yflat, evaluate=False)     # all passes local
print('see commented cells above for the explicit / staged Y_flat workflows')

## 9. Inspect results

In [ ]:
# Streamed mean projection of mc.zarr (RAM ~ batch * H * W * 8 bytes) for overlay.
def mean_projection(z, batch=500):
    T = z.shape[0]
    acc = np.zeros(z.shape[1:], dtype=np.float64)
    for s in range(0, T, batch):
        acc += np.asarray(z[s:min(s + batch, T)], dtype=np.float64).sum(axis=0)
    return (acc / T).astype(np.float32)

mc_mean = mean_projection(mc)

K = model.A.shape[1]
A_dense = np.asarray(model.A.todense()).reshape(H, W, K)
accepted = getattr(model, 'accepted_mask', np.ones(K, dtype=bool))

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(mc_mean, cmap='gray')
for k in range(K):
    fp = A_dense[..., k]
    if fp.max() <= 0:
        continue
    ax.contour(fp, levels=[fp.max() * 0.3],
               colors='red' if accepted[k] else '0.6', linewidths=0.7)
ax.set_title(f'{int(accepted.sum())} accepted / {K} total components')
ax.axis('off')
plt.tight_layout()
fig.savefig(MC_PLOTS / 'footprints_overlay.png', dpi=120)
plt.show()

In [ ]:
n_show = min(8, K)
if n_show == 0:
    print('no components to plot')
else:
    idx = np.where(accepted)[0]
    pool = idx if idx.size else np.arange(K)
    order = pool[np.argsort(-(model.C[pool]).max(axis=1))][:n_show]

    fig, axes = plt.subplots(n_show, 1, figsize=(11, 1.3 * n_show), sharex=True)
    if n_show == 1:
        axes = [axes]
    for ax, k in zip(axes, order):
        ax.plot(model.C[k] + model.YrA[k], color='0.6', lw=0.6, label='C + YrA')
        ax.plot(model.C[k], color='C3', lw=1.0, label='C (deconvolved)')
        ax.set_ylabel(f'#{k}')
    axes[0].legend(loc='upper right', fontsize=8)
    axes[-1].set_xlabel('frame')
    fig.suptitle('Top components — C+YrA (noisy projected) vs C (deconvolved)')
    plt.tight_layout()
    plt.show()

## 10. Notes

- The same `params_eff` ran MC and extraction; only the `yflat_*` / `n_jobs` /
  `bg_tsub` knobs in section 6 affect streaming speed (not results).
- To compare network vs local-SSD: run once with `YFLAT_DIR = None` (all-network)
  and once with a local `YFLAT_DIR`, and diff the per-stage timing summaries.
- Re-upsampling downsampled results to native resolution and the MC QC video are
  in `run_session.ipynb` (sections 7 / 10) — orthogonal to streaming.